# AI Code Assistant

### Context-Grounded Code Understanding System

##  System Overview

This project builds an AI Code Assistant designed to:

- Explain code snippets  
- Suggest fixes and improvements  
- Answer programming-related queries  

The system enhances LLM reasoning by grounding it with relevant code context retrieved dynamically.

##  System Architecture

The system follows a context-grounded reasoning pipeline:

User Query  
→ Code Representation (CodeBERT)  
→ Context Retrieval (FAISS)  
→ Relevance Filtering  
→ Context Construction  
→ Structured Reasoning Prompt  
→ DeepSeek LLM  
→ Final Response  

Retrieval is used as a supporting mechanism to improve reasoning quality.

## Step 1 — Install Libraries

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:

!pip install -q --upgrade pip
!pip install -q \
torch==2.10.0 \
transformers==4.51.3 \
accelerate==0.30.1 \
faiss-cpu \
einops

/Users/metafolder/.zshenv:1: % not found
/Users/metafolder/.zshenv:1: % not found


In [4]:
!pip install --force-reinstall --no-cache-dir numpy==1.26.4

/Users/metafolder/.zshenv:1: % not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 6.0 MB/s  0:00:02m0:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3


## Step 2 — Imports & GPU Check

In [5]:
import torch
import faiss
import pickle
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModel,
    BitsAndBytesConfig
)

print('✅ Imports done!')
print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name(0)}')
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM          : {total:.1f} GB')

ModuleNotFoundError: No module named 'torch'

## Step 3 — Load CodeBERT (Embedding Model)

**What is CodeBERT?**
A model trained on code that converts any text or snippet into a 768-dimensional vector.
Snippets with similar meaning produce similar vectors — this powers semantic search.

In [ ]:
print('Loading CodeBERT (~500 MB, ~2 min)...')

cb_tok = AutoTokenizer.from_pretrained('microsoft/codebert-base')
cb_mod = AutoModel.from_pretrained('microsoft/codebert-base')
cb_mod.eval()

if torch.cuda.is_available():
    cb_mod = cb_mod.cuda()

print('✅ CodeBERT ready!')


def get_embedding(text):
    """
    Convert text/code → 768-dim normalized vector using CodeBERT.
    Uses [CLS] token which summarises the full input sequence.
    """
    tokens = cb_tok(
        text,
        return_tensors='pt',
        max_length=512,
        truncation=True,
        padding='max_length'
    )
    if torch.cuda.is_available():
        tokens = {k: v.cuda() for k, v in tokens.items()}

    with torch.no_grad():
        output = cb_mod(**tokens)

    # CLS token = index 0
    vec = output.last_hidden_state[:, 0, :].cpu().numpy().squeeze()

    # Normalize for cosine similarity
    return vec / (np.linalg.norm(vec) + 1e-10)


# Quick test
v = get_embedding('def hello(): print("hi")')
print(f'Embedding shape: {v.shape}')  # (768,)

## Step 4 — Knowledge Base

**What is this?**
A small library of code snippets that the assistant references when answering.
This is what RAG **retrieves** from. In a real project, this could be your entire codebase.

In [ ]:
KNOWLEDGE_BASE = [
    {
        'title': 'Python List Comprehension',
        'code': (
            'numbers = [1, 2, 3, 4, 5, 6]\n'
            '# Get squares of even numbers\n'
            'result = [x**2 for x in numbers if x % 2 == 0]\n'
            'print(result)  # [4, 16, 36]'
        ),
        'note': 'Compact syntax for building lists. Faster and more Pythonic than a for-loop with append.'
    },
    {
        'title': 'Python Decorator',
        'code': (
            'import time\n'
            '\n'
            'def timer(func):\n'
            '    def wrapper(*args, **kwargs):\n'
            '        start = time.time()\n'
            '        result = func(*args, **kwargs)\n'
            '        print(f"Time: {time.time()-start:.3f}s")\n'
            '        return result\n'
            '    return wrapper\n'
            '\n'
            '@timer\n'
            'def slow_task():\n'
            '    time.sleep(1)\n'
            '\n'
            'slow_task()'
        ),
        'note': 'Decorators wrap a function to add behaviour before/after it runs. Use @syntax to apply.'
    },
    {
        'title': 'Binary Search',
        'code': (
            'def binary_search(arr, target):\n'
            '    left, right = 0, len(arr) - 1\n'
            '    while left <= right:\n'
            '        mid = (left + right) // 2\n'
            '        if arr[mid] == target: return mid\n'
            '        elif arr[mid] < target: left = mid + 1\n'
            '        else: right = mid - 1\n'
            '    return -1\n'
            '\n'
            'print(binary_search([1,3,5,7,9], 7))  # 3'
        ),
        'note': 'O(log n) search on sorted arrays. Cuts the search space in half every step.'
    },
    {
        'title': 'Error Handling Try-Except',
        'code': (
            'def safe_divide(a, b):\n'
            '    try:\n'
            '        return a / b\n'
            '    except ZeroDivisionError:\n'
            '        print("Cannot divide by zero!")\n'
            '        return None\n'
            '    except TypeError as e:\n'
            '        print(f"Type error: {e}")\n'
            '        return None\n'
            '    finally:\n'
            '        print("Done")\n'
            '\n'
            'print(safe_divide(10, 2))   # 5.0\n'
            'print(safe_divide(10, 0))   # None'
        ),
        'note': 'try-except catches errors. finally always runs. Catch specific exceptions before generic ones.'
    },
    {
        'title': 'Python Class (OOP)',
        'code': (
            'class BankAccount:\n'
            '    def __init__(self, owner, balance=0):\n'
            '        self.owner = owner\n'
            '        self.balance = balance\n'
            '\n'
            '    def deposit(self, amount):\n'
            '        if amount > 0:\n'
            '            self.balance += amount\n'
            '\n'
            '    def withdraw(self, amount):\n'
            '        if amount > self.balance:\n'
            '            raise ValueError("Insufficient funds")\n'
            '        self.balance -= amount\n'
            '\n'
            '    def __str__(self):\n'
            '        return f"{self.owner}: ${self.balance}"\n'
            '\n'
            'acc = BankAccount("Alice", 100)\n'
            'acc.deposit(50)\n'
            'print(acc)  # Alice: $150'
        ),
        'note': '__init__ is the constructor. __str__ controls how the object prints. self refers to the instance.'
    },
    {
        'title': 'Merge Sort',
        'code': (
            'def merge_sort(arr):\n'
            '    if len(arr) <= 1:\n'
            '        return arr\n'
            '    mid = len(arr) // 2\n'
            '    left  = merge_sort(arr[:mid])\n'
            '    right = merge_sort(arr[mid:])\n'
            '    result, i, j = [], 0, 0\n'
            '    while i < len(left) and j < len(right):\n'
            '        if left[i] <= right[j]:\n'
            '            result.append(left[i]); i += 1\n'
            '        else:\n'
            '            result.append(right[j]); j += 1\n'
            '    return result + left[i:] + right[j:]\n'
            '\n'
            'print(merge_sort([5, 2, 8, 1, 9]))  # [1,2,5,8,9]'
        ),
        'note': 'Divide-and-conquer sort. O(n log n) time, O(n) space. Stable sort.'
    },
    {
        'title': 'Generator with Yield',
        'code': (
            'def fibonacci():\n'
            '    a, b = 0, 1\n'
            '    while True:\n'
            '        yield a\n'
            '        a, b = b, a + b\n'
            '\n'
            'gen = fibonacci()\n'
            'first_10 = [next(gen) for _ in range(10)]\n'
            'print(first_10)  # [0,1,1,2,3,5,8,13,21,34]'
        ),
        'note': 'yield pauses function and returns value. Generators compute lazily — save memory.'
    },
    {
        'title': 'Dictionary Operations',
        'code': (
            'person = {"name": "Alice", "age": 25, "city": "Mumbai"}\n'
            '\n'
            '# Safe access with default\n'
            'print(person.get("salary", 0))   # 0 (no KeyError)\n'
            '\n'
            '# Add / update\n'
            'person["age"] = 26\n'
            '\n'
            '# Dict comprehension\n'
            'lengths = {k: len(str(v)) for k, v in person.items()}\n'
            '\n'
            '# Loop\n'
            'for key, val in person.items():\n'
            '    print(f"{key}: {val}")'
        ),
        'note': 'Dicts store key-value pairs. .get() avoids KeyError. Dict comprehensions are concise.'
    },
    {
        'title': 'Lambda Map Filter',
        'code': (
            'nums = [1, 2, 3, 4, 5, 6, 7, 8]\n'
            '\n'
            '# lambda: one-line anonymous function\n'
            'square = lambda x: x ** 2\n'
            '\n'
            '# map: apply function to every item\n'
            'squared = list(map(square, nums))\n'
            '\n'
            '# filter: keep items where True\n'
            'evens = list(filter(lambda x: x % 2 == 0, nums))\n'
            '\n'
            'print(squared)   # [1,4,9,16,25,36,49,64]\n'
            'print(evens)     # [2,4,6,8]'
        ),
        'note': 'lambda creates quick inline functions. map applies to all items. filter keeps matching ones.'
    },
    {
        'title': 'File Read and Write',
        'code': (
            '# Write\n'
            'with open("data.txt", "w") as f:\n'
            '    f.write("Hello\\n")\n'
            '    f.write("World\\n")\n'
            '\n'
            '# Read all\n'
            'with open("data.txt", "r") as f:\n'
            '    content = f.read()\n'
            '    print(content)\n'
            '\n'
            '# Read line by line\n'
            'with open("data.txt", "r") as f:\n'
            '    for line in f:\n'
            '        print(line.strip())'
        ),
        'note': 'Use with so file closes automatically. Modes: r=read, w=write, a=append.'
    }
]

print(f'✅ Knowledge base: {len(KNOWLEDGE_BASE)} snippets')
for i, item in enumerate(KNOWLEDGE_BASE, 1):
    print(f'   [{i:02d}] {item["title"]}')

## Step 5 — Build FAISS Index (Vector Database)

**What is FAISS?**
A fast library by Meta to search through vectors instantly.
We embed every snippet with CodeBERT, store those vectors in FAISS.
When you ask a question, FAISS finds the most similar snippets in milliseconds.

In [ ]:
print('Building FAISS vector index...')

# IndexFlatIP = exact cosine similarity
faiss_index      = faiss.IndexFlatIP(768)
metadata_store   = []
embeddings_store = []

for item in KNOWLEDGE_BASE:
    # Combine all text for a richer embedding
    full_text = f"{item['title']} {item['code']} {item['note']}"
    vec = get_embedding(full_text).astype('float32')

    embeddings_store.append(vec)
    metadata_store.append(item)
    print(f"  ✔ {item['title']}")

# Add all vectors at once
faiss_index.add(np.array(embeddings_store).astype('float32'))

print(f'\n✅ FAISS index ready!')
print(f'   {faiss_index.ntotal} vectors stored, dimension = 768')

## Step 6 — Retrieval Function (The R in RAG)

Embeds your question with CodeBERT, then searches FAISS for the most relevant snippets.

In [ ]:
def retrieve(query, top_k=3):
    """Find top-k most relevant code snippets for a query."""
    q_vec = get_embedding(query).astype('float32').reshape(1, -1)
    scores, indices = faiss_index.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            item = metadata_store[idx].copy()
            item['score'] = float(score)
            results.append(item)
    return results


# ── Test retrieval ──────────────────────────────────────────
print('Testing retrieval...\n')
test_queries = [
    'how to sort a list in Python',
    'handle errors in Python',
    'create a class with methods',
]
for q in test_queries:
    print(f'Query: "{q}"')
    for r in retrieve(q):
        bar = '█' * int(r['score'] * 20)
        print(f'  [{r["score"]:.3f}] {bar:<20} {r["title"]}')
    print()

## Step 7 — Load DeepSeek Coder (The LLM)

**What is DeepSeek Coder?**
A powerful open-source code-specialized LLM by DeepSeek AI.
We use the **1.3B Instruct** version — fits in free Colab T4 VRAM with 4-bit quantization.


In [ ]:
print('Loading DeepSeek Coder 1.3B Instruct...')
print('(4-bit quantized to save VRAM — fits on free Colab T4)\n')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DEEPSEEK_MODEL = 'deepseek-ai/deepseek-coder-1.3b-instruct'

# ✅ Check GPU
use_gpu = torch.cuda.is_available()

# ✅ 4-bit config ONLY if GPU
bnb_config = None
if use_gpu:
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        print("✅ Using 4-bit quantization")
    except:
        print("⚠️ bitsandbytes not available, loading normally")
        bnb_config = None

# ✅ Load tokenizer
ds_tok = AutoTokenizer.from_pretrained(
    DEEPSEEK_MODEL,
    trust_remote_code=True
)

# ✅ Load model safely
if use_gpu and bnb_config is not None:
    ds_mod = AutoModelForCausalLM.from_pretrained(
        DEEPSEEK_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    # CPU fallback (no quantization)
    ds_mod = AutoModelForCausalLM.from_pretrained(
        DEEPSEEK_MODEL,
        torch_dtype=torch.float32,
        device_map=None,
        trust_remote_code=True
    )

ds_mod.eval()

print('\n✅ DeepSeek Coder loaded!')

# ✅ GPU stats
if use_gpu:
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   VRAM used : {used:.1f} GB / {total:.1f} GB')
else:
    print("⚠️ Running on CPU (slower)")

## Step 8 — Conversation Memory

Stores your last few messages so DeepSeek knows the conversation context.

In [ ]:
class Memory:
    """Stores last N conversation turns for context."""

    def __init__(self, max_turns=6):
        self.turns = []
        self.max   = max_turns

    def save(self, role, text):
        self.turns.append({'role': role, 'text': text})
        if len(self.turns) > self.max * 2:
            self.turns = self.turns[-(self.max * 2):]

    def get(self):
        return self.turns

    def clear(self):
        self.turns = []
        print('🗑️  Memory cleared')

    def show(self):
        print(f'\n📝 Memory ({len(self.turns)} messages):')
        for t in self.turns:
            who = '👤 You' if t['role'] == 'user' else '🤖 Bot'
            print(f'  {who}: {t["text"][:70]}...')


memory = Memory(max_turns=6)
print('✅ Memory initialized')

## 🔹 Step 9 — End-to-End Query Processing Pipeline

Each user query is processed through three core stages:

### 1. Retrieve — Context Identification
Relevant code snippets are retrieved using:
- CodeBERT for embedding generation  
- FAISS for semantic similarity search  

This step identifies context that is closely related to the user’s query.

---

### 2. Augment — Context Construction
A structured prompt is built using:
- Retrieved code snippets  
- User query  
- Instruction format  
- Conversation history (if available)  

This ensures the model receives complete and relevant input.

---

### 3. Generate — Response Generation
The DeepSeek model processes the constructed prompt and generates:
- Explanation  
- Fixes or improvements  
- Structured output  

The model uses both its internal knowledge and the provided context.

---

### 🔧 Prompt Format

DeepSeek follows a structured chat format:

In [ ]:
def ask(query, mode='answer', top_k=3, max_tokens=400, temperature=0.3):
    import time
    t0 = time.time()

    print(f'\n{"═"*62}')
    print(f'  Mode: {mode.upper():10}  |  DeepSeek Coder')
    print(f'  Query: {query[:50]}...')
    print(f'═'*62)

    # ── Step 1: RETRIEVE ──────────────────────────────────
    print('\n🔍 [1/3] Retrieving with CodeBERT + FAISS...')
    snippets = retrieve(query, top_k=top_k)
    for s in snippets:
        print(f'        ✔ [{s["score"]:.3f}] {s["title"]}')

    # ── Step 2: AUGMENT ───────────────────────────────────
    print('\n📝 [2/3] Building augmented prompt...')
    prompt = build_prompt(query, snippets, mode, memory.get())

    # Tokenize
    inputs = ds_tok(
        prompt,
        return_tensors='pt',
        max_length=2048,
        truncation=True
    )

    n_tok = inputs["input_ids"].shape[1]
    print(f'        Prompt = {n_tok} tokens')

    # ✅ FIX: Always match model device
    device = next(ds_mod.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # ── Step 3: GENERATE ──────────────────────────────────
    print('\n🧠 [3/3] Generating with DeepSeek Coder...')

    with torch.no_grad():
        out = ds_mod.generate(
            **inputs,
            max_new_tokens=150,
            temperature=temperature,
            do_sample=False,   # ✅ cleaner output
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=ds_tok.eos_token_id
        )

    new_ids = out[0][inputs['input_ids'].shape[1]:]

    # Clean decoding
    answer = ds_tok.decode(new_ids, skip_special_tokens=True)
    answer = answer.replace("\u0120", " ").replace("\u010a", "\n").strip()

    # Save memory
    memory.save('user', query)
    memory.save('assistant', answer)

    elapsed = time.time() - t0
    print(f'        Done in {elapsed:.1f}s | {len(new_ids)} tokens')

    print(f'\n{"─"*62}')
    print('✅ ANSWER:')
    print('─'*62)
    print(answer)
    print('─'*62)

    return answer

## Step 10 — Test: EXPLAIN Mode

In [ ]:
ask(
    'Explain how this code works:\n\n'
    'result = [x**2 for x in range(10) if x % 2 == 0]\n'
    'print(result)',
    mode='explain'
)

## Step 10 — Test: FIX Mode

In [ ]:
ask(
    'Fix the bugs in this code:\n\n'
    'def get_average(numbers):\n'
    '    total = 0\n'
    '    for n in numbers:\n'
    '        total += n\n'
    '    return total / len(numbers)\n\n'
    'print(get_average([]))     # crashes with ZeroDivisionError\n'
    'print(get_average([1,2,3]))',
    mode='fix'
)

## Step 10 — Test: IMPROVE Mode

In [ ]:
ask(
    'Improve this code:\n\n'
    'def get_evens(lst):\n'
    '    result = []\n'
    '    for i in range(len(lst)):\n'
    '        if lst[i] % 2 == 0:\n'
    '            result.append(lst[i])\n'
    '    return result',
    mode='improve'
)

## Step 10 — Test: Q&A Mode

In [ ]:
ask(
    'What is the difference between a list and a tuple in Python?',
    mode='answer'
)

## Step 11 — Save Artifacts to Pickle

In [ ]:
import os

# Save FAISS index
faiss.write_index(faiss_index, '/content/faiss_index.bin')
print('✅ Saved: faiss_index.bin')

# Save metadata
with open('/content/metadata_store.pkl', 'wb') as f:
    pickle.dump(metadata_store, f)
print('✅ Saved: metadata_store.pkl')

# Save knowledge base
with open('/content/knowledge_base.pkl', 'wb') as f:
    pickle.dump(KNOWLEDGE_BASE, f)
print('✅ Saved: knowledge_base.pkl')

print('\nFile sizes:')
for name in ['faiss_index.bin', 'metadata_store.pkl', 'knowledge_base.pkl']:
    size = os.path.getsize(f'/content/{name}') / 1024
    print(f'  {name:<30} {size:.1f} KB')

## Step 12 — Download Files

In [ ]:
from google.colab import files

for path in [
    '/content/faiss_index.bin',
    '/content/metadata_store.pkl',
    '/content/knowledge_base.pkl'
]:
    print(f'📥 Downloading {path.split("/")[-1]}...')
    files.download(path)

print('\n✅ All files downloaded!')
print('\nNext steps for Streamlit:')
print('  1. Create folder:  my_project/artifacts/')
print('  2. Move 3 files → artifacts/')
print('  3. pip install -r requirements.txt')
print('  4. streamlit run app.py')

## Step 13 — Interactive Chat in Colab (Optional)
Run this cell for a live chat session inside the notebook.

In [ ]:
print('🤖 AI Code Assistant — Chat Mode (DeepSeek Coder)')
print('─'*55)
print('Commands:')
print('  mode:explain | mode:fix | mode:improve | mode:answer')
print('  clear    → clear memory')
print('  history  → show conversation history')
print('  quit     → exit')
print('─'*55)

current_mode = 'answer'

while True:
    try:
        user_input = input(f'\n[{current_mode}] You: ').strip()
    except EOFError:
        break

    if not user_input:
        continue

    if user_input.lower() == 'quit':
        print('Goodbye! 👋')
        break
    elif user_input.lower() == 'clear':
        memory.clear()
    elif user_input.lower() == 'history':
        memory.show()
    elif user_input.lower().startswith('mode:'):
        m = user_input.split(':', 1)[1].strip()
        if m in ['explain', 'fix', 'improve', 'answer']:
            current_mode = m
            print(f'✅ Mode → {current_mode}')
        else:
            print('Valid modes: explain | fix | improve | answer')
    else:
        ask(user_input, mode=current_mode)